# 02 · Portfolios

Calendar-time quintile portfolios: breakpoints from the prior year's distribution, entry the month after release, 3-month holding (the paper's Table II). See `filinglens.research.portfolios` for the timing rules.

In [ ]:
import pandas as pd

from filinglens.config import get_settings
from filinglens.research.prices import add_characteristics, load_panel
from filinglens.research.signals import load_signals
from filinglens.research.synthetic import make_world

RESEARCH = get_settings().processed_dir / "research"
REAL = all(
    (RESEARCH / f).exists() for f in ("signals.parquet", "prices.parquet", "factors.parquet")
)
if REAL:
    signals = load_signals(RESEARCH / "signals.parquet")
    panel = add_characteristics(load_panel(RESEARCH / "prices.parquet"))
    factors = pd.read_parquet(RESEARCH / "factors.parquet")
    factors.index = pd.PeriodIndex(factors.pop("month"), freq="M")
else:
    print("No research data yet: using the SYNTHETIC world (pipeline check, not results).")
    world = make_world()
    signals, panel, factors = world.signals, add_characteristics(world.panel), world.factors
source = panel.attrs.get("source")
print(f"{signals['cik'].nunique()} firms, {len(signals)} signal rows, returns: {source}")

In [ ]:
from filinglens.research.portfolios import PortfolioSpec, portfolio_returns

spec = PortfolioSpec(signal="sim_cosine_tf", section="full", hold_months=3, weighting="ew")
port = portfolio_returns(signals, panel, spec)
port[["Q1", "Q2", "Q3", "Q4", "Q5", "LS"]].describe().T[["mean", "std", "count"]].mul(
    [1e4, 1e4, 1]
).round(1)

In [ ]:
ax = (port["LS"].cumsum() * 100).plot(title=spec.label())
ax.axvline(pd.Period("2015-01", "M"), color="grey", linestyle="--")
ax.set_ylabel("cumulative L/S, % (non-compounded)")

Holdings per quintile and turnover. Thin quintiles make the L/S noisy, so check these before reading the alphas.

In [ ]:
port[[c for c in port if c.startswith("n_Q")] + ["turnover"]].describe().round(2)

In [ ]:
for w in ("ew", "vw"):
    p = portfolio_returns(signals, panel, spec.with_(weighting=w))
    print(w, f"L/S mean {p['LS'].mean() * 1e4:.1f} bps/month")